# Session 3: a river that changes, and a boat that corrects itself
**A. Real-world problem.** Real rivers flow fastest in the middle and slowly near the banks,
and wind pushes the boat as well. A fixed heading that is perfect for a uniform current
misses the landing. A ferry pilot looks at where the boat is drifting and steers back
towards the landing line: that is *feedback*. Today you simulate it in small time steps and
check the simulation against the exact results of Session 2.

This is a simplified educational model (flat 2D water, no waves, no boat inertia): it is not
a navigation tool.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
WIDTH = 60.0                       # river width (m)
MAX_HEADING = np.radians(60.0)     # the rudder limit: the boat never points more than 60 degrees off

## Step 1: the model
State: x (downstream, m) and y (across, m), starting at (0, 0). The landing target is the
point (0, W) straight opposite. The boat has speed v through the water and heading theta
(radians, positive = leaning upstream). Then

    dx/dt = u(y) + wind - v sin(theta)      dy/dt = v cos(theta)

* `u(y)` is the current. *Uniform*: u = umax. *Parabolic*: u(y) = 4 umax (y/W)(1 - y/W),
  zero at both banks and umax in the middle; its average across the river is (2/3) umax.
* `wind` is a steady sideways push in m/s, positive downstream.

In [ ]:
def current_profile(y, umax, profile="parabolic"):
    if profile == "uniform":
        return umax
    s = y / WIDTH
    return 4.0 * umax * s * (1.0 - s)

## Step 2: the heading controller
*Naive*: keep one fixed heading (`heading0_deg`) all the way. *Feedback*: every step look at
the cross-track error x (how far downstream of the target line we are) and steer upstream:

    theta = heading0 + K x + Ki * (sum of x dt),   limited to +-60 degrees

K is in degrees per metre of error; the second (integral) term has Ki = K / 20 and removes
the leftover error that a steady wind or current would otherwise cause. With K = 0 the
boat is naive.

In [ ]:
def simulate(v, umax, wind=0.0, K=0.0, heading0_deg=0.0, profile="parabolic", dt=0.05, method="rk4"):
    """Time-step the boat until it reaches y = WIDTH.
    Returns a dict with t, x, y, heading_deg (arrays) and landing_x (m), time (s)."""
    ki = K / 20.0
    h0 = np.radians(heading0_deg)

    def rate(x, y, th):
        return np.array([current_profile(y, umax, profile) + wind - v * np.sin(th), v * np.cos(th)])

    x = y = t = integ = 0.0
    T, X, Y, H = [0.0], [0.0], [0.0], []
    for _ in range(int(2000 / dt)):
        th = float(np.clip(h0 + np.radians(K) * x + np.radians(ki) * integ, -MAX_HEADING, MAX_HEADING))
        if method == "euler":
            nxt = np.array([x, y]) + dt * rate(x, y, th)
        else:
            s = np.array([x, y])
            k1 = rate(*s, th)
            k2 = rate(*(s + dt / 2 * k1), th)
            k3 = rate(*(s + dt / 2 * k2), th)
            k4 = rate(*(s + dt * k3), th)
            nxt = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        H.append(np.degrees(th))
        if nxt[1] >= WIDTH:                      # crossed the far bank inside this step
            f = (WIDTH - y) / (nxt[1] - y)
            x, t, y = x + f * (nxt[0] - x), t + f * dt, WIDTH
            T.append(t); X.append(x); Y.append(y)
            break
        x, y = nxt
        t += dt
        integ += x * dt
        T.append(t); X.append(x); Y.append(y)
    else:
        raise RuntimeError("the boat did not reach the far bank")
    H.append(H[-1])
    return {"t": np.array(T), "x": np.array(X), "y": np.array(Y), "heading_deg": np.array(H),
            "landing_x": float(X[-1]), "time": float(T[-1])}

## Step 3: the exact answers to compare with
For a *fixed* heading the boat moves along a straight line in the uniform case, and in the
parabolic case the across motion is still uniform (dy/dt = v cos theta), so the time is
W / (v cos theta) and the drift is the average current times the time:

    drift = (u_avg + wind - v sin theta) W / (v cos theta),   u_avg = umax or (2/3) umax.

In [ ]:
def analytic_fixed(v, umax, wind=0.0, heading0_deg=0.0, profile="parabolic"):
    """(landing_x, time) for a fixed heading, from the formulas above."""
    th = np.radians(heading0_deg)
    u_avg = umax if profile == "uniform" else 2.0 * umax / 3.0
    t = WIDTH / (v * np.cos(th))
    return float((u_avg + wind - v * np.sin(th)) * t), float(t)


def best_possible_drift(v, u):
    """Smallest |drift| (m) with a fixed heading in a uniform current u (exact, Session 2)."""
    return 0.0 if u < v else float(WIDTH * np.sqrt(u * u - v * v) / v)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    v, umax, wind = 4.0, 3.0, 0.5
    naive = simulate(v, umax, wind, K=0.0)
    open_loop = simulate(v, umax, wind, K=0.0, heading0_deg=float(np.degrees(np.arcsin(2.0 / 4.0))))
    fb = simulate(v, umax, wind, K=4.0)
    for name, r in [("naive straight", naive), ("fixed 30 deg", open_loop), ("feedback K=4", fb)]:
        print(f"{name:15s} lands {r['landing_x']:7.2f} m downstream after {r['time']:5.2f} s")
    ex, et = analytic_fixed(4.0, 3.0, 0.5, 0.0)
    print(f"analytic straight heading: {ex:.2f} m after {et:.2f} s (simulation {naive['landing_x']:.2f} m)")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name, r, col in [("naive: straight across", naive, "tab:red"), ("fixed 30 degrees upstream", open_loop, "tab:orange"),
                         ("feedback, K = 4 deg/m", fb, "tab:green")]:
        ax.plot(r["x"], r["y"], color=col, lw=2.5, label=f"{name}: lands {r['landing_x']:.1f} m")
    ax.axvline(0, color="k", ls=":", label="target line")
    ax.axhline(0, color="k")
    ax.axhline(WIDTH, color="k")
    ax.set_xlabel("distance downstream (m)")
    ax.set_ylabel("distance across the river (m)")
    ax.set_title("Boat 4 m/s, parabolic current (peak 3 m/s), wind 0.5 m/s")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "paths_compared.png", dpi=120)

    # validation: simulation against the exact formulas over many headings
    heads = np.arange(-40, 61, 5)
    err = [abs(simulate(v, umax, wind, heading0_deg=h)["landing_x"] - analytic_fixed(v, umax, wind, h)[0]) for h in heads]
    print(f"largest |simulation - exact| over {len(heads)} headings: {max(err):.2e} m")
    # sensitivity to the current
    peaks = np.linspace(0, 6, 25)
    naive_drift = [simulate(v, p, 0.0)["landing_x"] for p in peaks]
    fb_err = [simulate(v, p, 0.0, K=4.0)["landing_x"] for p in peaks]
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    ax2.plot(peaks, naive_drift, lw=2, color="tab:red", label="naive straight heading")
    ax2.plot(peaks, fb_err, lw=2, color="tab:green", label="feedback, K = 4 deg/m")
    ax2.set_xlabel("peak current (m/s)")
    ax2.set_ylabel("landing point, downstream (m)")
    ax2.set_title("Sensitivity to the current (boat 4 m/s, no wind)")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "sensitivity.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. Mini-project: plan a ferry route.

## Coding tasks
1. **Easy.** Print the landing point and time of `simulate(4, 3, 0.5, K=k)` for k = 0, 2 and 4, and describe in two sentences how the answer changes.
2. **Stretch.** Halve the time step in `simulate` from 0.05 to 0.025 and show that the landing point of the feedback boat changes by less than 0.1 m, then repeat with `method="euler"` for the fixed heading at dt = 0.5 and dt = 0.25 and compare the errors against 30.0 m.
3. **Explain.** Explain the failure case: why does a 4 m/s boat in a uniform 5 m/s current drift at least 45 m, whatever the heading? Which heading gives that minimum?

## Assessment
- **A. Conceptual.** Why does the feedback pilot use an integral term (the running sum of the error) as well as the proportional term K x?
- **B. Numerical.** Using the exact formula, find the drift of a 4 m/s boat that points straight across a 60 m river with a parabolic current of peak 5 m/s and no wind. Check it against the sensitivity graph.
- **C. Graph and simulation interpretation.** In the sensitivity graph, what is the feedback landing point for peak currents of 3, 5 and 6 m/s, and why does it get worse above 4 m/s?
- **D. Design and optimisation challenge.** Design: for the 4 m/s boat, the parabolic current with a 3 m/s peak and a 0.5 m/s wind, choose the gain K (in steps of 0.5) so that the landing is within 2 m and the crossing takes at most 21 s. Justify your choice.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    r = L.simulate(4.0, 3.0, 0.0)
    checks.append(("worked example: parabolic current peak 3 m/s, boat 4 m/s straight across: drift 30 m (= (2/3) 3 * 60 / 4) in 15 s",
                   abs(r["landing_x"] - 30.0) < 1e-3 and abs(r["time"] - 15.0) < 1e-3))
    r = L.simulate(4.0, 3.0, 0.5)
    checks.append(("adding a 0.5 m/s wind adds 0.5 * 15 = 7.5 m: 37.5 m", abs(r["landing_x"] - 37.5) < 1e-3))
    ru = L.simulate(4.0, 2.0, 0.0, profile="uniform")
    checks.append(("uniform current 2 m/s, straight heading: 30 m, as in Session 2", abs(ru["landing_x"] - 30.0) < 1e-3))
    ho = float(np.degrees(np.arcsin(2.0 / 4.0)))
    checks.append(("uniform current, heading 30 degrees: lands opposite, 17.32 s",
                   abs(L.simulate(4.0, 2.0, 0.0, heading0_deg=ho, profile="uniform")["landing_x"]) < 1e-3
                   and abs(L.simulate(4.0, 2.0, 0.0, heading0_deg=ho, profile="uniform")["time"] - 17.32) < 0.01))
    err = max(abs(L.simulate(4.0, 3.0, 0.5, heading0_deg=h)["landing_x"] - L.analytic_fixed(4.0, 3.0, 0.5, h)[0])
              for h in range(-40, 61, 10))
    checks.append(("simulation matches the exact fixed-heading formula to within 1 mm at 11 headings", err < 1e-3))
    e1 = abs(L.simulate(4.0, 3.0, 0.0, dt=0.5, method="euler")["landing_x"] - 30.0)
    e2 = abs(L.simulate(4.0, 3.0, 0.0, dt=0.25, method="euler")["landing_x"] - 30.0)
    checks.append(("Euler error shrinks when the time step is halved", e2 < e1))
    a = L.simulate(4.0, 3.0, 0.5, K=4.0, dt=0.05)["landing_x"]
    b = L.simulate(4.0, 3.0, 0.5, K=4.0, dt=0.0125)["landing_x"]
    checks.append(("feedback result changes by less than 0.1 m when dt is divided by 4", abs(a - b) < 0.1))
    naive = L.simulate(4.0, 3.0, 0.5, K=0.0)
    fb = L.simulate(4.0, 3.0, 0.5, K=4.0)
    checks.append(("feedback K = 4 lands within 2 m while the naive heading misses by 37.5 m",
                   abs(fb["landing_x"]) < 2.0 and abs(naive["landing_x"]) > 37.0))
    checks.append(("the price of correcting: the crossing takes longer (20.8 s against 15 s)",
                   fb["time"] > naive["time"] + 5.0 and abs(fb["time"] - 20.79) < 0.05))
    checks.append(("the heading never exceeds the 60 degree rudder limit", float(np.max(np.abs(fb["heading_deg"]))) <= 60.0 + 1e-9))
    # sensitivity
    d2 = L.simulate(4.0, 2.0, 0.0)["landing_x"]
    d3 = L.simulate(4.0, 3.0, 0.0)["landing_x"]
    checks.append(("sensitivity: the naive drift grows 10 m for every extra 1 m/s of peak current", abs((d3 - d2) - 10.0) < 1e-3))
    # edge: nothing pushes the boat
    z = L.simulate(4.0, 0.0, 0.0, K=4.0)
    checks.append(("no current and no wind: feedback keeps the boat on the line (lands at 0)", abs(z["landing_x"]) < 1e-9))
    # failure case: current faster than the boat
    best = L.best_possible_drift(4.0, 5.0)
    s1 = L.simulate(4.0, 5.0, 0.0, profile="uniform")["landing_x"]
    s2 = L.simulate(4.0, 5.0, 0.0, K=4.0, profile="uniform")["landing_x"]
    s3 = L.simulate(4.0, 5.0, 0.0, heading0_deg=float(np.degrees(np.arcsin(4.0 / 5.0))), profile="uniform")["landing_x"]
    checks.append(("failure case: uniform current 5 m/s > boat 4 m/s: best drift is 45 m and no run beats it",
                   abs(best - 45.0) < 1e-9 and abs(s3 - 45.0) < 1e-3 and s1 >= 45.0 and s2 >= 45.0 - 1e-6))
    checks.append(("failure case: for current < boat speed the best drift is 0", L.best_possible_drift(4.0, 3.9) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
